# NeurIPS 2026 Smart Buildings — Submission Version 10

**Experiment:** Critical-terminal trim-and-respond fan-pressure reset. Instead of PID, model fitting, or predicting future zone temperatures, the policy trims each running AHU's duct pressure when terminal demand is mild and responds when zones request more airflow.

## Components and responsibilities

| Component | Purpose |
| --- | --- |
| `_build_topology`, `_flow_pair` | Resolve each building's zone-to-AHU map and optional matched flow sensors |
| `_zone_demand`, `_comfort_fallback` | Decode standardized zone temperature and validate live setpoints |
| `_set_terminal` | Set VAV damper and reheat and register pressure requests |
| `_airflow_deficit`, `_reset_fan_pressure` | Avoid startup false alarms, then trim/respond with pressure limits |
| `_control_ahu` | Coordinate running AHU cooling, zone terminals, and duct pressure |
| `_control_hot_water` | Start hot-water plant for genuine reheat requests only |
| `_idle_actions`, `reset`, `get_action` | Enforce action contract and reset per-AHU state between episodes |

All constants and physical units are named explicitly. One control step visits each AHU and zone once: O(AHUs + zones) time, O(AHUs) additional controller state. Flow telemetry is optional; when unavailable, the terminal-request logic still operates.

**Review fixes:** For the first two active steps, ignore AHU flow readings that can be stale from pre-start. Comfort guard requests more pressure for significant unsatisfied zone errors before saturation; a severe violation always triggers a response. Pressure is bounded from 200 to 1600 Pa and resets when an AHU stops.

**Limitations:** A commanded VAV opening is only a proxy for terminal airflow. This is not a calibrated or fully compliant Guideline 36 sequence. Tests verify interface/behavior on synthetic inputs, not simulated comfort, electricity cost, carbon, or the leaderboard score. V7's known public score of 0.03947 is not yet surpassed by evidence.

**Synthetic tests:** `python Co_Kaggle/g8/submissions/test_version10.py`. Full simulator and Kaggle evaluations remain necessary.


In [ ]:
"""Version 10: demand-driven trim-and-respond fan-pressure control.

The experiment saves fan energy by gradually *trimming* each running AHU's duct
pressure. A near-open VAV that still needs conditioning (or a persistent,
measured AHU flow shortfall) requests a faster *response* in pressure. This is
not a PID, thermal predictor, online optimizer, or fully compliant Guideline 36
implementation. The pressure state is independent for each AHU.

Units: zone temperatures and setpoints are kelvin; duct pressures are pascals;
VAV openings are fractions. Actions must be normalized to [-1, 1].
"""
import re

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """A metadata-driven controller implementing a pressure-reset experiment."""

    # Five-minute control interval, independent pressure state for each AHU.
    INITIAL_PRESSURE_PA = 700.0
    MIN_PRESSURE_PA = 200.0
    MAX_PRESSURE_PA = 1600.0
    PRESSURE_TRIM_PA = 25.0
    PRESSURE_RESPONSE_PA = 120.0

    START_CONDITIONING_ERROR_K = 0.25
    TERMINAL_REQUEST_ERROR_K = 0.60
    TERMINAL_REQUEST_OPENING = 0.88
    # Prevent long periods of trimming while a zone remains clearly distressed.
    COMFORT_GUARD_ERROR_K = 2.0
    COMFORT_GUARD_OPENING = 0.65
    SEVERE_ERROR_K = 4.0
    FLOW_DEFICIT_RATIO = 0.80
    FLOW_SENSOR_WARMUP_STEPS = 2

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        metadata = building_meta_data
        self.continuous_count = int(metadata["n_continuous_action"])
        self.discrete_count = int(metadata["n_discrete_action"])
        self.observation_index = {
            str(name): position
            for position, name in enumerate(metadata["observation_field_names"])
        }
        self.supply_indices = list(map(int, metadata["temp_indices"]))
        self.fan_indices = list(map(int, metadata["pressure_indices"]))
        self.recirc_indices = list(map(int, metadata["recirculation_indices"]))
        water_indices = list(map(int, metadata["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_run_index = self.discrete_count - 1
        self.water_supply_index = water_indices[0] if water_indices else None
        self.water_pressure_index = (
            self.fan_indices[self.ahu_count]
            if len(self.fan_indices) > self.ahu_count else None
        )

        self.comfort_band = tuple(map(float, metadata["comfort_temp_window"]))
        self.eco_band = tuple(map(float, metadata["eco_temp_window"]))
        stats = metadata.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {}
        )
        self.zone_mean = float(stats.get("sample_mean", 295.0))
        self.zone_std = float(stats.get("sample_std", 1.0))
        if not np.isfinite(self.zone_mean):
            self.zone_mean = 295.0
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        self.ahus = self._build_topology(metadata)
        self.reset()

    # --- 1. Topology: resolve dynamic field/action indices once per building. ---
    @staticmethod
    def _numeric_order(name):
        match = re.search(r"\d+", str(name))
        return (int(match.group()) if match else -1, str(name))

    def _field(self, name):
        return self.observation_index.get(name)

    def _flow_pair(self, ahu_name):
        """Find only a matched sensor/setpoint pair, never unrelated fields.

        Flow measurements are optional. The AHU name and suffix are checked
        together so no fuzzy name match can silently select the wrong unit.
        """
        for prefix in (f"ahs_{ahu_name}_", f"{ahu_name}_"):
            for stem in ("supply_air_flowrate", "supply_air_flow_rate"):
                sensed = self._field(prefix + stem + "_sensor")
                target = self._field(prefix + stem + "_setpoint")
                if sensed is not None and target is not None:
                    return sensed, target
        return None, None

    def _build_topology(self, metadata):
        """Build AHU -> VAV mappings using the public building metadata."""
        ahus = []
        assignments = metadata["ahu_zone_assignments"]
        zone_actions = metadata["zone_action_indices"]
        for ahu_id, ahu_name in enumerate(sorted(assignments, key=self._numeric_order)):
            if ahu_id >= self.ahu_count:
                break
            zones = []
            for room in assignments[ahu_name]:
                vav_name = str(room).replace("room_", "VAV_", 1)
                temperature_index = self._field(
                    f"{vav_name}_zone_air_temperature_sensor"
                )
                if temperature_index is None:
                    continue
                actions = zone_actions.get(vav_name, {})
                zones.append({
                    "temperature": temperature_index,
                    "heat_setpoint": self._field(
                        f"{vav_name}_zone_air_heating_temperature_setpoint_sensor"
                    ),
                    "cool_setpoint": self._field(
                        f"{vav_name}_zone_air_cooling_temperature_setpoint_sensor"
                    ),
                    "damper_action": actions.get("supply_air_damper_percentage_command"),
                    "reheat_action": actions.get("reheat_valve_percentage_command"),
                })
            measured_flow, requested_flow = self._flow_pair(str(ahu_name))
            ahus.append({
                "id": ahu_id,
                "supply_action": self.supply_indices[ahu_id],
                "fan_action": (
                    self.fan_indices[ahu_id] if ahu_id < len(self.fan_indices) else None
                ),
                "recirc_action": (
                    self.recirc_indices[ahu_id]
                    if ahu_id < len(self.recirc_indices) else None
                ),
                "flow_sensor": measured_flow,
                "flow_target": requested_flow,
                "zones": zones,
            })
        return ahus

    # --- 2. Sensor decoding and normalised-action encoding. ---
    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, opening):
        """Map a physical [0, 1] opening to the required [-1, 1] action."""
        return 2.0 * cls._clamp(opening, 0.0, 1.0) - 1.0

    @classmethod
    def _supply_action(cls, kelvin):
        """Supply setpoint 285..305 kelvin -> normalized [-1, 1]."""
        return cls._clamp((kelvin - 295.0) / 10.0, -1.0, 1.0)

    @staticmethod
    def _read(observation, index):
        if index is None or not 0 <= index < len(observation):
            return float("nan")
        return float(observation[index])

    def _write(self, actions, index, value):
        if index is not None and 0 <= index < len(actions):
            actions[index] = self._clamp(value, -1.0, 1.0)

    def _comfort_fallback(self, observation):
        """Select the published comfort/eco band if live setpoints are invalid."""
        comfort_now = self._read(
            observation, self._field("comfort_mode_now")
        )
        return (self.comfort_band if np.isfinite(comfort_now) and comfort_now >= 0.5
                else self.eco_band)

    def _zone_demand(self, observation, zone, fallback):
        """Return heating/cooling shortfall in K, or None for bad temperature."""
        normalized = self._read(observation, zone["temperature"])
        temperature_k = self.zone_mean + self.zone_std * normalized
        if not np.isfinite(temperature_k) or not 240 <= temperature_k <= 340:
            return None
        heat_setpoint = self._read(observation, zone["heat_setpoint"])
        cool_setpoint = self._read(observation, zone["cool_setpoint"])
        if not (np.isfinite(heat_setpoint) and np.isfinite(cool_setpoint)
                and 245 <= heat_setpoint < cool_setpoint <= 335
                and cool_setpoint - heat_setpoint <= 35):
            heat_setpoint, cool_setpoint = fallback
        return {
            "zone": zone,
            "heating_error": max(0.0, heat_setpoint - temperature_k),
            "cooling_error": max(0.0, temperature_k - cool_setpoint),
        }

    # --- 3. VAV terminals: issue requests based on unsatisfied demand. ---
    def _set_terminal(self, demand, actions, heating_requests):
        """Command one VAV; return whether this zone requests more fan pressure."""
        heating = demand["heating_error"]
        cooling = demand["cooling_error"]
        zone = demand["zone"]
        damper = 0.10
        valve = 0.0

        if cooling >= self.START_CONDITIONING_ERROR_K:
            damper = self._clamp(0.17 + 0.27 * cooling, 0.17, 1.0)
        elif heating >= self.START_CONDITIONING_ERROR_K:
            damper = self._clamp(0.28 + 0.20 * heating, 0.28, 1.0)
            reheat_idx = zone["reheat_action"]
            if reheat_idx is not None and 0 <= reheat_idx < self.continuous_count:
                valve = self._clamp(0.18 + 0.28 * heating, 0.18, 1.0)
                heating_requests.append((heating, valve))

        self._write(actions, zone["damper_action"], self._fraction_action(damper))
        self._write(actions, zone["reheat_action"], self._fraction_action(valve))

        error = max(heating, cooling)
        saturated = (error >= self.TERMINAL_REQUEST_ERROR_K
                     and damper >= self.TERMINAL_REQUEST_OPENING)
        comfort_guard = (error >= self.COMFORT_GUARD_ERROR_K
                         and damper >= self.COMFORT_GUARD_OPENING)
        return saturated or comfort_guard

    # --- 4. Pressure: trim slowly, respond faster, per active AHU. ---
    def _airflow_deficit(self, ahu, observation):
        """Use AHU airflow feedback only after startup readings have settled."""
        ahu_id = ahu["id"]
        if self.running_steps[ahu_id] < self.FLOW_SENSOR_WARMUP_STEPS:
            return False
        sensed = self._read(observation, ahu["flow_sensor"])
        target = self._read(observation, ahu["flow_target"])
        return bool(np.isfinite(sensed) and np.isfinite(target)
                    and 0 <= sensed and target > 0
                    and sensed < self.FLOW_DEFICIT_RATIO * target)

    def _reset_fan_pressure(self, ahu, observation, terminal_requests, severe):
        """Bounded trim-and-respond update in physical pressure (Pa)."""
        ahu_id = ahu["id"]
        requests = terminal_requests + int(self._airflow_deficit(ahu, observation))
        if severe:
            requests = max(requests, 1)
        pressure_change = (
            self.PRESSURE_RESPONSE_PA * min(requests, 2)
            if requests else -self.PRESSURE_TRIM_PA
        )
        self.pressure_pa[ahu_id] = self._clamp(
            self.pressure_pa[ahu_id] + pressure_change,
            self.MIN_PRESSURE_PA, self.MAX_PRESSURE_PA
        )
        self.running_steps[ahu_id] += 1
        return self.pressure_pa[ahu_id]

    def _stop_ahu(self, ahu_id):
        """Discard stale pressure/flow state when an AHU stops."""
        self.pressure_pa[ahu_id] = self.INITIAL_PRESSURE_PA
        self.running_steps[ahu_id] = 0

    # --- 5. Plant dispatch: AHU cooling + shared hot-water VAV reheat. ---
    def _control_ahu(self, ahu, observation, demands, actions, switches,
                     heating_requests):
        """Run an AHU only when any of its valid zones needs conditioning."""
        ahu_id = ahu["id"]
        if not demands:
            self._stop_ahu(ahu_id)
            return
        peak_error = max(max(d["heating_error"], d["cooling_error"])
                         for d in demands)
        if peak_error < self.START_CONDITIONING_ERROR_K:
            self._stop_ahu(ahu_id)
            return

        switches[ahu_id] = 1
        peak_cooling = max(d["cooling_error"] for d in demands)
        # AHU cooling coil cannot heat: VAV reheat provides zone heating.
        supply_k = (self._clamp(298.0 - 1.8 * peak_cooling, 285.0, 305.0)
                    if peak_cooling else 305.0)
        self._write(actions, ahu["supply_action"], self._supply_action(supply_k))
        self._write(actions, ahu["recirc_action"], 1.0)  # Simulator caps at 70%.

        requests = sum(self._set_terminal(d, actions, heating_requests)
                       for d in demands)
        pressure_pa = self._reset_fan_pressure(
            ahu, observation, requests, peak_error >= self.SEVERE_ERROR_K
        )
        # Physical 0..2000 Pa -> normalized [-1, 1].
        self._write(actions, ahu["fan_action"], pressure_pa / 1000.0 - 1.0)

    def _control_hot_water(self, heating_requests, actions, switches):
        """Only start the central water loop for valid VAV reheat requests."""
        if not heating_requests or not 0 <= self.water_run_index < self.discrete_count:
            return
        switches[self.water_run_index] = 1
        worst_error = max(error for error, _ in heating_requests)
        mean_valve = sum(valve for _, valve in heating_requests) / len(heating_requests)
        water_k = self._clamp(323.0 + 4.0 * worst_error, 320.0, 348.0)
        self._write(actions, self.water_supply_index,
                    self._fraction_action((water_k - 310.0) / 40.0))
        self._write(actions, self.water_pressure_index,
                    self._fraction_action(self._clamp(0.40 + 0.50 * mean_valve,
                                                       0.40, 0.95)))

    def _idle_actions(self):
        """Normalized zero means half-open, not off: choose explicit idle."""
        actions = np.full(self.continuous_count, -1.0, dtype=np.float32)
        switches = np.zeros(self.discrete_count, dtype=np.int32)
        for index in self.supply_indices:
            self._write(actions, index, 1.0)  # Idle cooling coil.
        for index in self.recirc_indices:
            self._write(actions, index, 1.0)  # Maximum permitted return air.
        return actions, switches

    def reset(self):
        """Reset between episodes: no stale pressure or flow warm-up state."""
        self.pressure_pa = [self.INITIAL_PRESSURE_PA] * self.ahu_count
        self.running_steps = [0] * self.ahu_count

    def get_action(self, obs):
        """One O(AHUs + zones) five-minute control step, O(AHUs) state."""
        observation = np.asarray(obs, dtype=np.float64).reshape(-1)
        actions, switches = self._idle_actions()
        fallback = self._comfort_fallback(observation)
        heating_requests = []

        for ahu in self.ahus:
            demands = [self._zone_demand(observation, z, fallback)
                       for z in ahu["zones"]]
            valid_demands = [d for d in demands if d is not None]
            self._control_ahu(ahu, observation, valid_demands,
                              actions, switches, heating_requests)

        self._control_hot_water(heating_requests, actions, switches)
        np.nan_to_num(actions, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(actions, -1.0, 1.0, out=actions)
        return {"continuous_action": actions, "discrete_action": switches}


In [ ]:
# Official Kaggle notebook inference gateway.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
